In [1]:
import torch
from torch import nn
import torch.optim as optim
import torch.nn.functional as F

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [3]:
# define model parameters (same as original paper here)
NUM_EPOCHS = 90
BATCH_SIZE = 128
MOMENTUM = 0.9
LR_DECAY = 0.0005
LR_INIT = 0.01
IMAGE_DIM = 227 
NUM_CLASSES = 1000  # 1000 classes for imagenet 2012 dataset
DEVICE_IDS = [0, 1, 2, 3]  # GPUs to use


In [4]:
class ConvBlock(nn.Module):
  def __init__(self, in_channels, out_channels, **kwargs):
    super(ConvBlock, self).__init__()
    self.conv = nn.Conv2d(in_channels, out_channels, **kwargs)

  def forward(self, x):
    return self.conv(x)


In [5]:
class Alexnet(nn.Module):
  def __init__(self, num_classes = 1000):
    super(Alexnet, self).__init__()
    # Although paddings aren't specified in the paper, the given paddings should be followed to have the same dimensions as in the paper.
    self.conv1 = ConvBlock(in_channels=3, out_channels=96, kernel_size=11, stride=4, padding =2) # input size is 224x224, next layer is 55x55, so pdding must be 2
    self.norm = nn.LocalResponseNorm(size=5, alpha=0.0001, beta=0.75, k=2)
    self.pool = nn.MaxPool2d(kernel_size = 3, stride=2) # 55x55 -> 27x27
    self.conv2 = ConvBlock(in_channels=96, out_channels=256, kernel_size=5, padding = 2)
    self.conv3 = ConvBlock(in_channels=256, out_channels=384, kernel_size=3, padding = 1)
    self.conv4 = ConvBlock(in_channels=384, out_channels=384, kernel_size=3, padding = 1)
    self.conv5 = ConvBlock(in_channels=384, out_channels=256, kernel_size=3, padding = 1)

    self.fc1 = nn.Linear(6*6*256,4096)
    self.fc1.bias.data.fill_(1)
    self.fc2 = nn.Linear(4096, 4096)
    self.fc2.bias.data.fill_(1)
    self.out = nn.Linear(4096,num_classes)
    self.relu = nn.LeakyReLU()
    self.dropout = nn.Dropout(0.5)
    self.softmax = nn.Softmax(dim=1)


  def forward(self,x):
    x = self.conv1(x) # 224x224 -> 55x55
    x = self.relu(x)
    x = self.norm(x)
    x = self.pool(x)  # 55x55 -> 27x27

    x = self.conv2(x)  # 27x27 -> 27x27
    x = self.relu(x)
    x = self.norm(x)
    x = self.pool(x)   # 27x27 -> 13x13

    x = self.conv3(x)  # 13x13 -> 13x13
    x = self.relu(x)
    x = self.conv4(x)  # 13x13 -> 13x13
    x = self.relu(x)
    x = self.conv5(x)  # 13x13 -> 13x13
    x = self.relu(x)
    x = self.pool(x)   # 13x13 -> 6x6

    x = torch.flatten(x,1)
    x = self.dropout(x)

    x = self.fc1(x)
    x = self.relu(x)
    x = self.dropout(x)
    x = self.fc2(x)
    x = self.relu(x)
    x = self.out(x)
    #x = self.relu(x)
    x = self.softmax(x)
    return x


In [6]:
from torchvision.datasets import CIFAR10
from torchvision import transforms
from torch.utils.data.sampler import SubsetRandomSampler
import numpy as np

In [7]:

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), # based on imagenet statistics
])

In [8]:
# Load data
train_dataset = CIFAR10(root='./data', train=True, transform=transform, download=True)
test_dataset = CIFAR10(root='./data', train=False, transform=transform, download=True)

In [9]:
from torch.utils.data import DataLoader
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [10]:
import torch.optim as optim

In [11]:
model = Alexnet(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr = 0.001, momentum = MOMENTUM, weight_decay=LR_DECAY)
#scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.0001)

In [12]:
# Training

num_epochs = NUM_EPOCHS

for epoch in range(num_epochs):
  model.train()
  running_loss = 0.0
  for images,labels in train_loader:
    images = images.to(device)
    labels = labels.to(device)
    optimizer.zero_grad()
    outputs = model(images)
    loss = criterion(outputs,labels)
    loss.backward()
    optimizer.step()
    #scheduler.step()
    running_loss += loss.item()

  print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}")


Epoch [1/90], Loss: 2.3110
Epoch [2/90], Loss: 2.3078
Epoch [3/90], Loss: 2.3063
Epoch [4/90], Loss: 2.3054
Epoch [5/90], Loss: 2.3047
Epoch [6/90], Loss: 2.3044
Epoch [7/90], Loss: 2.2759
Epoch [8/90], Loss: 2.1925
Epoch [9/90], Loss: 2.1754
Epoch [10/90], Loss: 2.1692
Epoch [11/90], Loss: 2.1648
Epoch [12/90], Loss: 2.1501
Epoch [13/90], Loss: 2.1272
Epoch [14/90], Loss: 2.1018
Epoch [15/90], Loss: 2.0794
Epoch [16/90], Loss: 2.0604
Epoch [17/90], Loss: 2.0452
Epoch [18/90], Loss: 2.0309
Epoch [19/90], Loss: 2.0202
Epoch [20/90], Loss: 2.0075
Epoch [21/90], Loss: 1.9904
Epoch [22/90], Loss: 1.9770
Epoch [23/90], Loss: 1.9634
Epoch [24/90], Loss: 1.9525
Epoch [25/90], Loss: 1.9446
Epoch [26/90], Loss: 1.9272
Epoch [27/90], Loss: 1.9151
Epoch [28/90], Loss: 1.9043
Epoch [29/90], Loss: 1.8916
Epoch [30/90], Loss: 1.8805
Epoch [31/90], Loss: 1.8697
Epoch [32/90], Loss: 1.8593
Epoch [33/90], Loss: 1.8479
Epoch [34/90], Loss: 1.8382
Epoch [35/90], Loss: 1.8313
Epoch [36/90], Loss: 1.8207
E

In [13]:
# Evaluation
model.eval()
total, correct = 0, 0

with torch.no_grad():
  for images, labels in test_loader:
    images, labels = images.to(device), labels.to(device)
    outputs = model(images)
    _, predicted = torch.max(outputs,1)
    total += labels.size(0)
    correct += (labels==predicted).sum().item()

accuracy = 100*correct/total
print(f"Accuracy: {accuracy: .2f}%")

Accuracy:  82.13%
